# Used Car Price Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge

# Find the training file without depending on one working directory.
_train_candidates = [
    Path("./content/dataset-2.csv"),
    Path("./datasets/dataset-2.csv"),
    Path("../data/dataset-2.csv"),
    Path("../datasets/dataset-2.csv"),
    Path("./dataset-2.csv"),
]
TRAIN_PATH = next((str(p) for p in _train_candidates if p.exists()), str(_train_candidates[0]))
TARGET_COLUMN = "price"

# Keep the starter default, but also support the data/datasets layout used by the task folder.
_eval_candidates = [
    Path("./evaluation_split_task2.csv"),
    Path("./data/evaluation_split_task2.csv"),
    Path("./datasets/evaluation_split_task2.csv"),
    Path("../data/evaluation_split_task2.csv"),
    Path("../datasets/evaluation_split_task2.csv"),
]
EVALUATION_SPLIT_PATH = next(
    (str(p) for p in _eval_candidates if p.exists()),
    "./evaluation_split_task2.csv",
)


## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [15]:
df = pd.read_csv(TRAIN_PATH)
print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (3207, 12)
      brand                   model  model_year      milage fuel_type  \
0       BMW                  M235 i        2015  81,000 mi.  Gasoline   
1       BMW                   750 i        2017  24,650 mi.  Gasoline   
2     Aston  Martin V8 Vantage Base        2008  62,378 mi.  Gasoline   
3  INFINITI               QX80 Luxe        2022  49,860 mi.  Gasoline   
4    Subaru             BRZ Premium        2023   5,800 mi.  Gasoline   

                                              engine transmission  \
0  320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...  6-Speed M/T   
1       445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel          A/T   
2       380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel          M/T   
3                               5.6L V8 32V GDI DOHC    Automatic   
4       228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel  6-Speed M/T   

          ext_col   int_col                                accident  \
0           White       Red                      

## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and extreme price/mileage outliers.
- Document any rows/columns you drop and why.

In [16]:
def _price_to_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[$,]", "", regex=True),
        errors="coerce",
    )


def data_cleaning(df):
    df = df.copy()

    # Standardize missing text values and surrounding spaces.
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    # The supplied CSV uses the column name "milage".
    mileage_col = None
    if "milage" in df.columns:
        mileage_col = "milage"
    elif "mileage" in df.columns:
        mileage_col = "mileage"

    if mileage_col is not None:
        df["mileage_num"] = pd.to_numeric(
            df[mileage_col].astype(str).str.replace(r"[^0-9.]", "", regex=True),
            errors="coerce",
        )
        df = df.drop(columns=[mileage_col])

    return df


def remove_bad_training_rows(df):
    # This function is used only on data that is currently being used for fitting.
    result = df.copy()
    result = result.drop_duplicates()

    if TARGET_COLUMN in result.columns:
        prices = _price_to_numeric(result[TARGET_COLUMN])
        valid = prices.notna() & (prices > 0)
        result = result.loc[valid].copy()

    return result


## 3. Feature Engineering

Transform the raw columns into features that are more useful for a regression model.

For example: calculating vehicle age from `model_year`, extracting horsepower/displacement from `engine`, or categorizing rare brands.

In [17]:
def feature_engineering(df):
    df = df.copy()

    if "model_year" in df.columns:
        df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce")
        df["vehicle_age"] = 2026 - df["model_year"]

    if "engine" in df.columns:
        engine_text = df["engine"].astype(str)

        df["horsepower"] = pd.to_numeric(
            engine_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)\s*HP", expand=False),
            errors="coerce",
        )
        df["engine_liters"] = pd.to_numeric(
            engine_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)\s*L\b", expand=False),
            errors="coerce",
        )

        cylinder_text = engine_text.str.extract(
            r"([0-9]+)\s*Cylinder", expand=False
        )
        v_engine_text = engine_text.str.extract(r"\bV([0-9]+)\b", expand=False)
        straight_engine_text = engine_text.str.extract(
            r"\bStraight\s*([0-9]+)\b", expand=False
        )
        cylinder_text = cylinder_text.fillna(v_engine_text).fillna(straight_engine_text)
        df["cylinders"] = pd.to_numeric(cylinder_text, errors="coerce")

    if "accident" in df.columns:
        df["accident_flag"] = (
            df["accident"]
            .astype(str)
            .str.contains("At least 1", case=False, na=False)
            .astype(int)
        )

    if "clean_title" in df.columns:
        df["clean_title_flag"] = (
            df["clean_title"].astype(str).str.lower().eq("yes").astype(int)
        )

    if "brand" in df.columns and "model" in df.columns:
        df["brand_model"] = (
            df["brand"].fillna("Unknown").astype(str)
            + "__"
            + df["model"].fillna("Unknown").astype(str)
        )

    return df


## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables (e.g., One-Hot or Target Encoding).
- Scale/normalize continuous numerical features (e.g., `mileage`, `age`, engine specs).

In [18]:
def scale_features(df):
    # Fitted imputation, encoding, and scaling are intentionally kept inside
    # the model Pipeline so validation/test/hidden rows cannot influence them.
    return df.copy()


In [19]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [20]:
def split_data(df):
    train_split, remaining_split = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
    )
    validation_split, test_split = train_test_split(
        remaining_split,
        test_size=0.50,
        random_state=42,
    )
    return train_split, validation_split, test_split


# Split the raw dataframe before fitting any preprocessing step.
train_split, validation_split, test_split = split_data(df)
train_split = remove_bad_training_rows(train_split)

print("Train shape:", train_split.shape)
print("Validation shape:", validation_split.shape)
print("Test shape:", test_split.shape)


Train shape: (2244, 12)
Validation shape: (481, 12)
Test shape: (482, 12)


## 6. Model Training, and Hyperparameter Tuning

- Train regression models on the train set (e.g., Ridge, Random Forest, XGBoost/LightGBM).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 7) to report **train** and **validation**
  metrics for each model: RMSE, MAE, R².
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_price(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, and returns price predictions from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that has not been touched yet.

In [21]:
def _make_model(regressor, feature_frame):
    numeric_features = feature_frame.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [
        col for col in feature_frame.columns if col not in numeric_features
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=2,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )

    # Training in log space makes the model less sensitive to very expensive cars.
    # TransformedTargetRegressor automatically converts predictions back to dollars.
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def _prepare_xy(split):
    X = split.drop(columns=[TARGET_COLUMN])
    y = _price_to_numeric(split[TARGET_COLUMN])
    valid = y.notna() & (y > 0)
    X = X.loc[valid].copy()
    y = y.loc[valid].copy()
    X = preprocess_data(X)
    return X, y


X_train, y_train = _prepare_xy(train_split)
X_validation, y_validation = _prepare_xy(validation_split)

candidate_configs = {
    "ExtraTrees_A": ExtraTreesRegressor(
        n_estimators=180,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees_B": ExtraTreesRegressor(
        n_estimators=180,
        min_samples_leaf=1,
        max_features=0.70,
        random_state=42,
        n_jobs=-1,
    ),
    "Ridge": Ridge(alpha=10.0),
}

validation_results = []
trained_candidates = {}

for name, regressor in candidate_configs.items():
    candidate = _make_model(regressor, X_train)
    candidate.fit(X_train, y_train)
    trained_candidates[name] = candidate

    train_pred = candidate.predict(X_train)
    validation_pred = candidate.predict(X_validation)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)
    train_r2 = r2_score(y_train, train_pred)

    validation_rmse = np.sqrt(mean_squared_error(y_validation, validation_pred))
    validation_mae = mean_absolute_error(y_validation, validation_pred)
    validation_r2 = r2_score(y_validation, validation_pred)

    validation_results.append(
        {
            "model": name,
            "train_RMSE": train_rmse,
            "train_MAE": train_mae,
            "train_R2": train_r2,
            "validation_RMSE": validation_rmse,
            "validation_MAE": validation_mae,
            "validation_R2": validation_r2,
        }
    )

results_df = pd.DataFrame(validation_results).sort_values("validation_MAE")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["model"]
print("Selected model:", best_model_name)

# Refit the selected configuration using train + validation only.
train_validation_split = pd.concat(
    [train_split, validation_split],
    axis=0,
    ignore_index=True,
)
train_validation_split = remove_bad_training_rows(train_validation_split)
X_train_validation, y_train_validation = _prepare_xy(train_validation_split)

best_regressor = candidate_configs[best_model_name]
model = _make_model(best_regressor, X_train_validation)
model.fit(X_train_validation, y_train_validation)


def predict_price(df):
    processed = preprocess_data(df)
    predictions = model.predict(processed)
    return np.maximum(np.asarray(predictions, dtype=float), 0.0)


       model   train_RMSE    train_MAE  train_R2  validation_RMSE  validation_MAE  validation_R2
ExtraTrees_B 3.014560e-09 1.185644e-09  1.000000     14937.774087     7265.058762       0.854642
ExtraTrees_A 3.014560e-09 1.185644e-09  1.000000     15051.418569     7324.999747       0.852422
       Ridge 8.342443e+04 1.223936e+04  0.242454     19448.271232     8640.176567       0.753607
Selected model: ExtraTrees_B


## 7. Evaluate on the Held-Out Test Set

Run `predict_price()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [22]:
def evaluate_predictions(y_true, y_pred):
    # DO NOT modify
    # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [23]:
X_test_raw = test_split.drop(columns=[TARGET_COLUMN])
y_test = _price_to_numeric(test_split[TARGET_COLUMN])
valid_test = y_test.notna() & (y_test > 0)

X_test_raw = X_test_raw.loc[valid_test].copy()
y_test = y_test.loc[valid_test].copy()

test_predictions = predict_price(X_test_raw)
test_metrics = evaluate_predictions(y_test, test_predictions)
pd.DataFrame([test_metrics])


--- Metrics ---
      RMSE: 33349.5407
       MAE: 10896.0153
        R2: 0.6657


,RMSE,MAE,R2
0,33349.540657,10896.015334,0.665746


### Approach Summary

The raw dataset is split into train, validation, and test sets before any fitted preprocessing is performed. Missing text values are standardized, mileage is converted to a numeric value, and no rows are dropped by the cleaning or feature-engineering functions. Vehicle age, horsepower, engine displacement, cylinder count, accident status, clean-title status, and a brand-model feature are created. Numerical imputation/scaling and categorical imputation/one-hot encoding are fitted only inside a scikit-learn pipeline. Extra Trees and Ridge regression configurations are compared using the validation split, with validation MAE used to select the final model. The selected model is refitted on train plus validation data and evaluated once on the untouched test split. Price is learned in log space and automatically transformed back to US dollars for `predict_price()`.


---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])